# Deep Ensemble GNN Training — Google Colab

Self-contained notebook to train 5 PointNetTransfGAT models (seeds 42, 137, 256, 389, 512).

## Before running:
1. Upload the training data folder `dist_not_connected_10k_1pct/` to your Google Drive (keep it as-is, 20 .pt files).
2. Set the `DRIVE_DATA_PATH` variable in Cell 2 to the correct path on your Drive.
3. Runtime → Change runtime type → **GPU (A100 or T4)**.
4. Run all cells in order. Each seed trains sequentially. After each seed finishes, model is saved to Drive.

## After training:
Download `model.pth` for each seed and place at:
`code/data/TR-C_Benchmarks/deep_ensemble_seed{SEED}/trained_model/model.pth`

Then run `python evaluate_deep_ensemble.py` from project root.

In [ ]:
# Cell 1: Install dependencies
# Detects installed PyTorch version automatically — no hardcoded version.
# Restart runtime after this cell if torch_geometric was freshly installed.
import subprocess, sys, torch

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-2000:])
    if result.returncode != 0: print('STDERR:', result.stderr[-1000:])

TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'Detected: PyTorch {TORCH_VER}, CUDA tag: {CUDA_TAG}')

print('Installing torch_geometric...')
run('pip install torch_geometric --quiet')

print('Installing sparse kernels...')
ok = subprocess.run(
    f'pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv '
    f'-f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html --quiet',
    shell=True, capture_output=True, text=True
)
if ok.returncode != 0:
    print('  Sparse kernels unavailable — falling back to pure-Python (slower but correct).')

run('pip install scikit-learn scipy --quiet')

import torch_geometric
print(f'torch_geometric {torch_geometric.__version__} ready.')
print('If freshly installed, restart runtime now and re-run from Cell 2.')


In [ ]:
# Cell 2: Configuration — EDIT THIS

# Path to the data folder on YOUR Google Drive
# Example: '/content/drive/MyDrive/thesis_data/dist_not_connected_10k_1pct'
DRIVE_DATA_PATH = '/content/drive/MyDrive/data/train_data/dist_not_connected_10k_1pct'

# Output folder on Drive where models will be saved
DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/deep_ensemble_models'

# Seeds for the deep ensemble (DO NOT CHANGE — must match thesis)
SEEDS = [42, 137, 256, 389, 512]

# T8 hyperparameters (DO NOT CHANGE)
HPARAMS = {
    'in_channels': 5,
    'out_channels': 1,
    'lr': 0.0005,
    'dropout': 0.2,
    'use_dropout': True,
    'batch_size': 8,
    'gradient_accumulation_steps': 3,
    'early_stopping_patience': 25,
    'num_epochs': 500,
    'loss_fct': 'mse',
    'use_gradient_clipping': True,
    'predict_mode_stats': False,
    'use_weighted_loss': False,
}

In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# Cell 4: Imports and device setup
import os
import copy
import math
import random
import time
from abc import ABC, abstractmethod
from enum import IntEnum

import numpy as np
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torch_geometric.nn import (
    Sequential as GeoSequential,
    TransformerConv, GATConv, PointNetConv, global_mean_pool
)
from torch_geometric.data import Batch

# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('WARNING: No GPU found. Training will be very slow on CPU.')

print(f'Using device: {device}')

In [ ]:
# Cell 5: EdgeFeatures enum (mirrors process_simulations_for_gnn.py)

class EdgeFeatures(IntEnum):
    VOL_BASE_CASE = 0
    CAPACITY_BASE_CASE = 1
    CAPACITY_REDUCTION = 2
    FREESPEED = 3
    LENGTH = 5  # Index 4 is HIGHWAY, 5 is LENGTH in original data

# The 5 features used for training (ablation study result)
NODE_FEATURES = ['VOL_BASE_CASE', 'CAPACITY_BASE_CASE', 'CAPACITY_REDUCTION', 'FREESPEED', 'LENGTH']
CONTINUOUS_FEAT = [EdgeFeatures.VOL_BASE_CASE, EdgeFeatures.CAPACITY_BASE_CASE,
                   EdgeFeatures.CAPACITY_REDUCTION, EdgeFeatures.FREESPEED, EdgeFeatures.LENGTH]

print('EdgeFeatures defined.')

In [ ]:
# Cell 6: Data loading and preprocessing

def set_random_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def collate_fn(data_list):
    return Batch.from_data_list(data_list)

def split_into_subsets(dataset, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, shuffle_seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9
    n = len(dataset)
    random.Random(shuffle_seed).shuffle(dataset)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    train_set = Subset(dataset, range(0, train_end))
    val_set   = Subset(dataset, range(train_end, val_end))
    test_set  = Subset(dataset, range(val_end, n))
    print(f'Split: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}')
    return train_set, val_set, test_set

def normalize_x_features(data_list, batch_size=100):
    scaler = StandardScaler()
    num_nodes = data_list[0].x.shape[0]
    for i in range(0, len(data_list), batch_size):
        batch = data_list[i:i+batch_size]
        batch_x = np.vstack([d.x[:, CONTINUOUS_FEAT].numpy() for d in batch])
        scaler.partial_fit(batch_x)
    for i in range(0, len(data_list), batch_size):
        batch = data_list[i:i+batch_size]
        batch_x = np.vstack([d.x[:, CONTINUOUS_FEAT].numpy() for d in batch])
        batch_x_norm = scaler.transform(batch_x)
        for j, d in enumerate(batch):
            d.x[:, CONTINUOUS_FEAT] = torch.tensor(
                batch_x_norm[j*num_nodes:(j+1)*num_nodes], dtype=d.x.dtype)
    # Filter to only the 5 selected features
    feat_idx = [EdgeFeatures[f].value for f in NODE_FEATURES]
    for d in data_list:
        d.x = d.x[:, feat_idx]
    return data_list, scaler

def normalize_pos_features(data_list, batch_size=1000):
    scaler = StandardScaler()
    num_nodes = data_list[0].x.shape[0]
    for i in range(0, len(data_list), batch_size):
        batch = data_list[i:i+batch_size]
        batch_pos = np.vstack([d.pos.numpy().reshape(-1, 6) for d in batch])
        scaler.partial_fit(batch_pos)
    for i in range(0, len(data_list), batch_size):
        batch = data_list[i:i+batch_size]
        for d in batch:
            pos_r = d.pos.numpy().reshape(-1, 6)
            pos_n = scaler.transform(pos_r)
            d.pos = torch.tensor(pos_n.reshape(num_nodes, 3, 2), dtype=d.pos.dtype)
    return data_list, scaler

def normalize_subset(subset):
    data_list = [copy.deepcopy(subset.dataset[idx]) for idx in subset.indices]
    print('  Normalizing x features...')
    data_list, x_scaler = normalize_x_features(data_list)
    print('  Normalizing pos features...')
    data_list, pos_scaler = normalize_pos_features(data_list)
    return data_list, {'x_scaler': x_scaler, 'pos_scaler': pos_scaler}

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print('Data utilities defined.')

In [ ]:
# Cell 7: Model architecture (PointNetTransfGAT + BaseGNN)
# Inlined from project source — DO NOT MODIFY

class LinearWarmupCosineDecayScheduler:
    def __init__(self, initial_lr, total_steps):
        self.initial_lr = initial_lr
        self.total_steps = total_steps
        self.min_lr = 0.01 * initial_lr
        self.warmup_steps = int(0.05 * total_steps)
        self.decay_steps = total_steps - self.warmup_steps
        self.cosine_decay_rate = 0.5

    def get_lr(self, step):
        if step < self.warmup_steps:
            return self.initial_lr * (step / max(1, self.warmup_steps))
        progress = (step - self.warmup_steps) / max(1, self.decay_steps)
        cosine_decay = self.cosine_decay_rate * (1 + math.cos(math.pi * progress))
        return self.min_lr + (self.initial_lr - self.min_lr) * cosine_decay


class GNN_Loss:
    def __init__(self, loss_fct_name, device, weighted=False):
        if loss_fct_name == 'mse':
            self.loss_fct = nn.MSELoss(reduction='mean').to(device)
        elif loss_fct_name == 'l1':
            self.loss_fct = nn.L1Loss(reduction='mean').to(device)
        else:
            raise ValueError(f'Unsupported loss: {loss_fct_name}')
        self.weighted = weighted

    def __call__(self, y_pred, y_true, x_unscaled=None):
        return self.loss_fct(y_pred, y_true)


class EarlyStopping:
    def __init__(self, patience=25, verbose=True):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss >= self.best_loss:
            self.counter += 1
            if self.verbose:
                print(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def compute_r2(preds, targets):
    mean_t = torch.mean(targets)
    ss_tot = torch.sum((targets - mean_t) ** 2)
    ss_res = torch.sum((targets - preds) ** 2)
    return (1 - ss_res / ss_tot).item()


def validate_model(model, valid_dl, loss_fct, device, scalers_validation):
    model.eval()
    val_loss = 0
    num_batches = 0
    all_preds = []
    all_targets = []
    with torch.inference_mode():
        for data in valid_dl:
            data = data.to(device)
            x_unscaled = scalers_validation['x_scaler'].inverse_transform(
                data.x.detach().clone().cpu().numpy())
            pred = model(data)
            val_loss += loss_fct(pred, data.y, x_unscaled).item()
            all_preds.append(pred)
            all_targets.append(data.y)
            num_batches += 1
    total_val_loss = val_loss / num_batches
    preds_cat = torch.cat(all_preds)
    targets_cat = torch.cat(all_targets)
    r2 = compute_r2(preds_cat, targets_cat)
    preds_np = preds_cat.cpu().numpy().flatten()
    targets_np = targets_cat.cpu().numpy().flatten()
    spearman, _ = spearmanr(preds_np, targets_np)
    pearson, _  = pearsonr(preds_np, targets_np)
    return total_val_loss, r2, spearman, pearson


class BaseGNN(nn.Module, ABC):
    def __init__(self, in_channels, out_channels, dropout=0.3, use_dropout=False,
                 predict_mode_stats=False, dtype=torch.float32):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.dropout = dropout
        self.use_dropout = use_dropout
        self.predict_mode_stats = predict_mode_stats
        self.dtype = dtype

    @abstractmethod
    def define_layers(self): pass

    @abstractmethod
    def forward(self, data): pass

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def train_one_seed(self, hparams, train_dl, valid_dl, scalers_train,
                       scalers_validation, model_save_path, checkpoint_dir, device):
        """
        Full training loop. Prints one summary line per epoch (no per-batch tqdm).
        Saves best model to model_save_path. Saves checkpoints every 20 epochs.
        Returns (best_val_loss, best_epoch).
        """
        scaler_amp = torch.amp.GradScaler(device.type, enabled=False)  # AMP off for stability
        total_steps = hparams['num_epochs'] * len(train_dl)
        scheduler = LinearWarmupCosineDecayScheduler(initial_lr=hparams['lr'], total_steps=total_steps)
        loss_fct = GNN_Loss(hparams['loss_fct'], device)
        optimizer = optim.AdamW(self.parameters(), lr=hparams['lr'], weight_decay=1e-4)
        early_stopping = EarlyStopping(patience=hparams['early_stopping_patience'], verbose=True)
        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

        best_val_loss = float('inf')
        best_epoch = 0

        for epoch in range(hparams['num_epochs']):
            super(BaseGNN, self).train()
            optimizer.zero_grad()
            epoch_train_loss = 0.0
            t0 = time.time()

            for idx, data in enumerate(train_dl):
                step = epoch * len(train_dl) + idx
                lr = scheduler.get_lr(step)
                for pg in optimizer.param_groups:
                    pg['lr'] = lr

                data = data.to(device)
                x_unscaled = scalers_train['x_scaler'].inverse_transform(
                    data.x.detach().clone().cpu().numpy())

                with torch.amp.autocast(device_type=device.type, enabled=False):
                    pred = self(data)
                    loss = loss_fct(pred, data.y, x_unscaled)

                epoch_train_loss += loss.item()
                scaler_amp.scale(loss).backward()


                if (idx + 1) % hparams['gradient_accumulation_steps'] == 0:
                    if hparams['use_gradient_clipping']:
                        nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                    scaler_amp.step(optimizer)
                    scaler_amp.update()
                    optimizer.zero_grad()

            # Flush remaining gradients
            if len(train_dl) % hparams['gradient_accumulation_steps'] != 0:
                if hparams['use_gradient_clipping']:
                    nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                scaler_amp.step(optimizer)
                scaler_amp.update()
                optimizer.zero_grad()

            # Validation
            val_loss, r2, spearman, pearson = validate_model(
                self, valid_dl, loss_fct, device, scalers_validation)

            elapsed = time.time() - t0
            avg_train_loss = epoch_train_loss / len(train_dl)
            print(f'Epoch {epoch+1:3d}/{hparams["num_epochs"]} | '
                  f'train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | '
                  f'R2={r2:.4f} | lr={lr:.6f} | {elapsed:.1f}s')

            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch
                torch.save(self.state_dict(), model_save_path)
                print(f'  --> Best model saved (val_loss={val_loss:.4f})')

            # Checkpoint every 20 epochs
            if epoch % 20 == 0:
                ckpt_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch}.pt')
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_val_loss': best_val_loss,
                    'val_loss': val_loss,
                }, ckpt_path)

            early_stopping(val_loss)
            if early_stopping.early_stop:
                print(f'Early stopping at epoch {epoch+1}.')
                break

        print(f'Training complete. Best val_loss={best_val_loss:.4f} at epoch {best_epoch+1}.')
        return best_val_loss, best_epoch


class PointNetTransfGAT(BaseGNN):
    def __init__(self, in_channels=5, out_channels=1,
                 point_net_conv_layer_structure_local_mlp=[256],
                 point_net_conv_layer_structure_global_mlp=[512],
                 gat_conv_layer_structure=[128, 256, 512],
                 dropout=0.3, use_dropout=False,
                 predict_mode_stats=False, dtype=torch.float32):
        super().__init__(in_channels=in_channels, out_channels=out_channels,
                         dropout=dropout, use_dropout=use_dropout,
                         predict_mode_stats=predict_mode_stats, dtype=dtype)
        self.pnc_local  = point_net_conv_layer_structure_local_mlp
        self.pnc_global = point_net_conv_layer_structure_global_mlp
        self.gat_conv   = gat_conv_layer_structure
        self.define_layers()
        self.initialize_weights()

    def define_layers(self):
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)
        self.point_net_conv_1 = self._create_pnc(is_first=True,  is_last=False)
        self.point_net_conv_2 = self._create_pnc(is_first=False, is_last=True)
        layers_global = self._define_gat_layers()
        self.gat_graph_layers = GeoSequential('x, edge_index', layers_global)
        self.gat_final = GATConv(64, 1)

    def forward(self, data):
        x = data.x.to(self.dtype)
        edge_index = data.edge_index
        pos1 = data.pos[:, 0, :]  # start position
        pos2 = data.pos[:, 1, :]  # end position
        x = self.point_net_conv_1(x, pos1, edge_index)
        x = self.point_net_conv_2(x, pos2, edge_index)
        x = self.gat_graph_layers(x, edge_index)
        return self.gat_final(x, edge_index)

    def _define_gat_layers(self):
        layers = []
        for idx in range(len(self.gat_conv) - 1):
            layers.append((TransformerConv(self.gat_conv[idx],
                                           int(self.gat_conv[idx+1]/4),
                                           heads=4), 'x, edge_index -> x'))
            layers.append(nn.ReLU(inplace=True))
            if self.use_dropout:
                layers.append(self.dropout_layer)
        layers.append((GATConv(self.gat_conv[-1], 64), 'x, edge_index -> x'))
        return layers

    def _create_pnc(self, is_first, is_last):
        offset = 2  # pos offset
        local_layers = []
        in_dim = (self.in_channels if is_first else self.pnc_global[-1]) + offset
        local_layers.append(nn.Linear(in_dim, self.pnc_local[0]))
        local_layers.append(nn.ReLU())
        if self.use_dropout:
            local_layers.append(self.dropout_layer)
        for i in range(len(self.pnc_local) - 1):
            local_layers.append(nn.Linear(self.pnc_local[i], self.pnc_local[i+1]))
            local_layers.append(nn.ReLU())
            if self.use_dropout:
                local_layers.append(self.dropout_layer)
        local_mlp = nn.Sequential(*local_layers)

        global_layers = []
        global_layers.append(nn.Linear(self.pnc_local[-1], self.pnc_global[0]))
        global_layers.append(nn.ReLU())
        if self.use_dropout:
            global_layers.append(self.dropout_layer)
        for i in range(len(self.pnc_global) - 1):
            global_layers.append(nn.Linear(self.pnc_global[i], self.pnc_global[i+1]))
            global_layers.append(nn.ReLU())
            if self.use_dropout:
                global_layers.append(self.dropout_layer)
        out_dim = self.gat_conv[0] if is_last else self.pnc_global[-1]
        global_layers.append(nn.Linear(self.pnc_global[-1], out_dim))
        global_layers.append(nn.ReLU())
        if self.use_dropout:
            global_layers.append(self.dropout_layer)
        global_mlp = nn.Sequential(*global_layers)

        return PointNetConv(local_nn=local_mlp, global_nn=global_mlp)

    def initialize_weights(self):
        super().initialize_weights()  # Linear layers
        for m in self.modules():
            if isinstance(m, PointNetConv):
                for name, param in m.local_nn.named_parameters():
                    if param.dim() > 1:
                        init.kaiming_normal_(param, mode='fan_out', nonlinearity='relu')
                    else:
                        init.zeros_(param)
                for name, param in m.global_nn.named_parameters():
                    if param.dim() > 1:
                        init.kaiming_normal_(param, mode='fan_out', nonlinearity='relu')
                    else:
                        init.zeros_(param)
            elif isinstance(m, GATConv):
                if hasattr(m, 'lin') and m.lin is not None:
                    init.xavier_normal_(m.lin.weight)
                    if m.lin.bias is not None:
                        init.zeros_(m.lin.bias)
                if hasattr(m, 'att_src') and m.att_src is not None:
                    init.xavier_normal_(m.att_src)
                if hasattr(m, 'att_dst') and m.att_dst is not None:
                    init.xavier_normal_(m.att_dst)

print('Model architecture defined.')
# Quick sanity check: instantiate and count params
_test_model = PointNetTransfGAT(
    in_channels=5, out_channels=1, dropout=0.2, use_dropout=True)
_num_params = sum(p.numel() for p in _test_model.parameters())
print(f'Model parameters: {_num_params:,}  (expected ~1,416,835)')
del _test_model

In [ ]:
# Cell 8: Load training data
# This takes a few minutes — 2.6 GB of .pt files

print(f'Loading data from: {DRIVE_DATA_PATH}')
datalist = []
batch_num = 1
while True:
    batch_file = os.path.join(DRIVE_DATA_PATH, f'datalist_batch_{batch_num}.pt')
    if not os.path.exists(batch_file):
        break
    print(f'  Loading batch {batch_num}...', end=' ')
    batch_data = torch.load(batch_file, map_location='cpu', weights_only=False)
    if isinstance(batch_data, list):
        datalist.extend(batch_data)
    print(f'done ({len(batch_data)} graphs)')
    batch_num += 1

print(f'Total graphs loaded: {len(datalist)}')

# Fix num_nodes
for data in datalist:
    data.num_nodes = data.x.shape[0]

print(f'Sample graph: x={datalist[0].x.shape}, pos={datalist[0].pos.shape}, y={datalist[0].y.shape}')

In [ ]:
# Cell 9: Split and normalize data
# shuffle_seed=42 is FIXED — same split for all 5 seeds (correct for deep ensemble)

print('Splitting dataset (shuffle_seed=42, fixed for all seeds)...')
train_set, val_set, test_set = split_into_subsets(datalist, shuffle_seed=42)

print('\nNormalizing train set...')
train_data_norm, scalers_train = normalize_subset(train_set)

print('\nNormalizing validation set...')
val_data_norm, scalers_val = normalize_subset(val_set)

print('\nCreating DataLoaders...')
train_dl = DataLoader(
    train_data_norm, batch_size=HPARAMS['batch_size'],
    shuffle=True, num_workers=2, pin_memory=True,
    collate_fn=collate_fn, worker_init_fn=seed_worker)

val_dl = DataLoader(
    val_data_norm, batch_size=HPARAMS['batch_size'],
    shuffle=False, num_workers=2, pin_memory=True,
    collate_fn=collate_fn)

print(f'Train batches: {len(train_dl)}, Val batches: {len(val_dl)}')
print('Data preparation complete.')

In [ ]:
# Cell 10: Train seed 42
SEED = 42
print(f'\n========== TRAINING SEED {SEED} ==========')
set_random_seeds(SEED)

model_dir = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'trained_model')
ckpt_dir  = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'checkpoints')
os.makedirs(model_dir, exist_ok=True)

model_42 = PointNetTransfGAT(
    in_channels=HPARAMS['in_channels'],
    out_channels=HPARAMS['out_channels'],
    dropout=HPARAMS['dropout'],
    use_dropout=HPARAMS['use_dropout'],
    predict_mode_stats=False,
    dtype=torch.float32
).to(device)

best_val_loss_42, best_epoch_42 = model_42.train_one_seed(
    hparams=HPARAMS,
    train_dl=train_dl,
    valid_dl=val_dl,
    scalers_train=scalers_train,
    scalers_validation=scalers_val,
    model_save_path=os.path.join(model_dir, 'model.pth'),
    checkpoint_dir=ckpt_dir,
    device=device
)
print(f'SEED {SEED} DONE: best_val_loss={best_val_loss_42:.4f}, best_epoch={best_epoch_42+1}')
del model_42
torch.cuda.empty_cache()

In [ ]:
# Cell 11: Train seed 137
SEED = 137
print(f'\n========== TRAINING SEED {SEED} ==========')
set_random_seeds(SEED)

model_dir = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'trained_model')
ckpt_dir  = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'checkpoints')
os.makedirs(model_dir, exist_ok=True)

model_137 = PointNetTransfGAT(
    in_channels=HPARAMS['in_channels'],
    out_channels=HPARAMS['out_channels'],
    dropout=HPARAMS['dropout'],
    use_dropout=HPARAMS['use_dropout'],
    predict_mode_stats=False,
    dtype=torch.float32
).to(device)

best_val_loss_137, best_epoch_137 = model_137.train_one_seed(
    hparams=HPARAMS,
    train_dl=train_dl,
    valid_dl=val_dl,
    scalers_train=scalers_train,
    scalers_validation=scalers_val,
    model_save_path=os.path.join(model_dir, 'model.pth'),
    checkpoint_dir=ckpt_dir,
    device=device
)
print(f'SEED {SEED} DONE: best_val_loss={best_val_loss_137:.4f}, best_epoch={best_epoch_137+1}')
del model_137
torch.cuda.empty_cache()

In [ ]:
# Cell 12: Train seed 256
SEED = 256
print(f'\n========== TRAINING SEED {SEED} ==========')
set_random_seeds(SEED)

model_dir = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'trained_model')
ckpt_dir  = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'checkpoints')
os.makedirs(model_dir, exist_ok=True)

model_256 = PointNetTransfGAT(
    in_channels=HPARAMS['in_channels'],
    out_channels=HPARAMS['out_channels'],
    dropout=HPARAMS['dropout'],
    use_dropout=HPARAMS['use_dropout'],
    predict_mode_stats=False,
    dtype=torch.float32
).to(device)

best_val_loss_256, best_epoch_256 = model_256.train_one_seed(
    hparams=HPARAMS,
    train_dl=train_dl,
    valid_dl=val_dl,
    scalers_train=scalers_train,
    scalers_validation=scalers_val,
    model_save_path=os.path.join(model_dir, 'model.pth'),
    checkpoint_dir=ckpt_dir,
    device=device
)
print(f'SEED {SEED} DONE: best_val_loss={best_val_loss_256:.4f}, best_epoch={best_epoch_256+1}')
del model_256
torch.cuda.empty_cache()

In [ ]:
# Cell 13: Train seed 389
SEED = 389
print(f'\n========== TRAINING SEED {SEED} ==========')
set_random_seeds(SEED)

model_dir = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'trained_model')
ckpt_dir  = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'checkpoints')
os.makedirs(model_dir, exist_ok=True)

model_389 = PointNetTransfGAT(
    in_channels=HPARAMS['in_channels'],
    out_channels=HPARAMS['out_channels'],
    dropout=HPARAMS['dropout'],
    use_dropout=HPARAMS['use_dropout'],
    predict_mode_stats=False,
    dtype=torch.float32
).to(device)

best_val_loss_389, best_epoch_389 = model_389.train_one_seed(
    hparams=HPARAMS,
    train_dl=train_dl,
    valid_dl=val_dl,
    scalers_train=scalers_train,
    scalers_validation=scalers_val,
    model_save_path=os.path.join(model_dir, 'model.pth'),
    checkpoint_dir=ckpt_dir,
    device=device
)
print(f'SEED {SEED} DONE: best_val_loss={best_val_loss_389:.4f}, best_epoch={best_epoch_389+1}')
del model_389
torch.cuda.empty_cache()

In [ ]:
# Cell 14: Train seed 512
SEED = 512
print(f'\n========== TRAINING SEED {SEED} ==========')
set_random_seeds(SEED)

model_dir = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'trained_model')
ckpt_dir  = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{SEED}', 'checkpoints')
os.makedirs(model_dir, exist_ok=True)

model_512 = PointNetTransfGAT(
    in_channels=HPARAMS['in_channels'],
    out_channels=HPARAMS['out_channels'],
    dropout=HPARAMS['dropout'],
    use_dropout=HPARAMS['use_dropout'],
    predict_mode_stats=False,
    dtype=torch.float32
).to(device)

best_val_loss_512, best_epoch_512 = model_512.train_one_seed(
    hparams=HPARAMS,
    train_dl=train_dl,
    valid_dl=val_dl,
    scalers_train=scalers_train,
    scalers_validation=scalers_val,
    model_save_path=os.path.join(model_dir, 'model.pth'),
    checkpoint_dir=ckpt_dir,
    device=device
)
print(f'SEED {SEED} DONE: best_val_loss={best_val_loss_512:.4f}, best_epoch={best_epoch_512+1}')
del model_512
torch.cuda.empty_cache()

In [ ]:
# Cell 15: Summary and verification
print('\n========== TRAINING SUMMARY ==========')
results = [
    (42,  best_val_loss_42,  best_epoch_42),
    (137, best_val_loss_137, best_epoch_137),
    (256, best_val_loss_256, best_epoch_256),
    (389, best_val_loss_389, best_epoch_389),
    (512, best_val_loss_512, best_epoch_512),
]
print(f'{"Seed":>6} | {"best_val_loss":>14} | {"best_epoch":>10} | {"file_size_MB":>12}')
print('-' * 55)
for seed, bvl, bep in results:
    model_path = os.path.join(DRIVE_OUTPUT_PATH, f'deep_ensemble_seed{seed}', 'trained_model', 'model.pth')
    size_mb = os.path.getsize(model_path) / 1e6 if os.path.exists(model_path) else -1
    status = 'OK' if 5.0 < size_mb < 10.0 else 'WARNING: size unexpected'
    print(f'{seed:>6} | {bvl:>14.4f} | {bep+1:>10} | {size_mb:>10.2f} MB  {status}')

print('\nAll models saved to:', DRIVE_OUTPUT_PATH)
print('\nNext steps:')
print('1. Download model.pth for each seed from Drive')
print('2. Place at: code/data/TR-C_Benchmarks/deep_ensemble_seed{SEED}/trained_model/model.pth')
print('3. Run: python evaluate_deep_ensemble.py')